In [0]:
dbutils.widgets.removeAll()

In [0]:
dbutils.widgets.text("catalogo", "catalog_dev")
dbutils.widgets.text("esquema_source", "bronze")
dbutils.widgets.text("esquema_sink", "silver")

dbutils.widgets.text("tabla_bronze", "customers_bronze")
dbutils.widgets.text("tabla_silver", "customers_silver")

catalogo = dbutils.widgets.get("catalogo")
esquema_source = dbutils.widgets.get("esquema_source")
esquema_sink = dbutils.widgets.get("esquema_sink")
tabla_bronze = dbutils.widgets.get("tabla_bronze")
tabla_silver = dbutils.widgets.get("tabla_silver")

In [0]:
df_bronze = spark.table(f"{catalogo}.{esquema_source}.{tabla_bronze}")

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

df_silver = df_bronze

str_cols = [
    field.name
    for field in df_silver.schema.fields
    if isinstance(field.dataType, StringType)
]

for c in str_cols:
    df_silver = df_silver.withColumn(c, trim(col(c)))


yes_no_map = create_map(
    lit("Yes"), lit(1),
    lit("No"), lit(0)
)

df_silver = df_silver \
    .withColumn("Ever_Married_Flag", yes_no_map[col("Ever_Married")]) \
    .withColumn("Graduated_Flag", yes_no_map[col("Graduated")])


df_silver = df_silver \
    .withColumn("Work_Experience", coalesce(col("Work_Experience"), lit(0.0))) \
    .withColumn("Family_Size", coalesce(col("Family_Size"), lit(1.0)))


df_silver = df_silver.withColumn(
    "Age_Group",
    when(col("Age") < 25, "Young")
    .when(col("Age") < 45, "Adult")
    .otherwise("Senior")
)

df_silver = df_silver.withColumn(
    "Spending_Level_Score",
    when(col("Spending_Score") == "Low", 1)
    .when(col("Spending_Score") == "Average", 2)
    .when(col("Spending_Score") == "High", 3)
    .otherwise(0)
)

df_silver = df_silver \
    .withColumn("ID", col("ID").cast(IntegerType())) \
    .withColumn("Age", col("Age").cast(IntegerType())) \
    .withColumn("Work_Experience", col("Work_Experience").cast(DoubleType())) \
    .withColumn("Family_Size", col("Family_Size").cast(DoubleType())) \
    .withColumn("Ever_Married_Flag", col("Ever_Married_Flag").cast(IntegerType())) \
    .withColumn("Graduated_Flag", col("Graduated_Flag").cast(IntegerType())) \
    .withColumn("Spending_Level_Score", col("Spending_Level_Score").cast(IntegerType()))

df_silver = df_silver.filter(col("ID").isNotNull())

In [0]:
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{catalogo}.{esquema_sink}.{tabla_silver}")

In [0]:
df_silver.printSchema()
df_silver.limit(5).display()